# Energy H Long CLP / MCTS Experiments

This notebook runs a compact CLP-style experiment on `electricity_H_long`, the hourly Energy config from GIFT-Eval.

The flow is:

1. Clone [`chahineNejm/kernels_playground`](https://github.com/chahineNejm/kernels_playground) and [`chahineNejm/graph_Time_series`](https://github.com/chahineNejm/graph_Time_series) in Colab.
2. Load a small slice of `electricity_H_long` using `kernels_playground/first_tests/utils`.
3. Convert the examples into arrays `H` and `F` for the graph CLP state.
4. Build the `graph_Time_series` grammar.
5. Run MCTS over cleaning -> feature -> model -> STOP chains.
6. Inspect the best chain, MCTS tree, and a graph/GIEN-style summary of what the search learned.

Default settings are intentionally small so the notebook can run interactively. Increase `N_STOP`, decrease `STEP`, raise `HISTORY_LEN`, or enable tree models once the loop is behaving.

## 0. Setup

This notebook is meant to run on Colab. The first code cell clones both GitHub repositories into `/content`, installs the kernel playground requirements, and adds the right package paths to `sys.path`.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

if IN_COLAB:
    %cd /content
    import subprocess
    import time

    def run_retry(cmd, attempts=3, delay=5):
        """Run a shell command with simple retry logic for Colab network hiccups."""
        last_exc = None
        for attempt in range(1, attempts + 1):
            print(f"[{attempt}/{attempts}]", " ".join(cmd))
            try:
                subprocess.check_call(cmd)
                return
            except subprocess.CalledProcessError as exc:
                last_exc = exc
                if attempt < attempts:
                    print(f"command failed; retrying in {delay}s...")
                    time.sleep(delay)
        raise last_exc

    def clone_or_update(repo_url, target):
        target = Path(target)
        if (target / ".git").exists():
            run_retry(["git", "-C", str(target), "pull", "--ff-only"])
        else:
            run_retry(["git", "clone", "--depth", "1", repo_url, str(target)])

    clone_or_update("https://github.com/chahineNejm/kernels_playground.git", "/content/kernels_playground")
    clone_or_update("https://github.com/chahineNejm/graph_Time_series.git", "/content/graph_Time_series")

    run_retry([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", "/content/kernels_playground/first_tests/requirements.txt",
        "scikit-learn", "networkx", "tqdm"
    ])

    KERNELS_REPO = Path("/content/kernels_playground")
    GRAPH_REPO = Path("/content/graph_Time_series")
else:
    # Local fallback for running this notebook from the graph construction folder.
    GRAPH_REPO = Path.cwd() / "graph_Time_series"
    KERNELS_REPO = Path.cwd().parent / "kernels_playground"

FIRST_TESTS = KERNELS_REPO / "first_tests"

# Import layout notes:
# - kernels_playground exposes utils from first_tests/utils, so add first_tests.
# - graph_Time_series may be a package directory itself; adding /content lets
#   `import graph_Time_series` work when cloned to /content/graph_Time_series.
# - adding GRAPH_REPO too also supports repo layouts with a nested package.
for p in [FIRST_TESTS, GRAPH_REPO.parent, GRAPH_REPO]:
    p = str(p.resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

print("IN_COLAB:", IN_COLAB)
print("KERNELS_REPO:", KERNELS_REPO, KERNELS_REPO.exists())
print("FIRST_TESTS:", FIRST_TESTS, FIRST_TESTS.exists())
print("GRAPH_REPO:", GRAPH_REPO, GRAPH_REPO.exists())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

from utils.config import DATASETS
from utils.data import build_examples
from utils.augmentation import uniform_length

from graph_Time_series import State, Grammar, plot_grammar, mcts_search, print_mcts_tree
from graph_Time_series.token import Token
from graph_Time_series.tokens.cleaning import (
    CleanIdentity,
    CleanDetrend,
    CleanMovingAvg,
    CleanNormalize,
    CleanDetrendNorm,
)
from graph_Time_series.tokens.features import FeatRaw, FeatFFTEncode, FeatLagFeatures
from graph_Time_series.tokens.models import ModelKernelRBF, ModelRandomForest, ModelXGBoost, StopToken, compute_mase

np.set_printoptions(precision=4, suppress=True)

## 1. Experiment Knobs

`electricity_H_long` can be large, and the graph search evaluates pipelines with leave-one-out model predictions. Start small, then scale up.

For now the notebook runs **all enabled combinations exhaustively**. MCTS is kept as an optional later section because it is useful once the grammar gets larger.

Current policy:

- normalization is mandatory and always happens first
- detrending is an optional second cleaning step after normalization
- model sequences fit residuals automatically through `State.current_target`

Notes:

- `HISTORY_LEN=512` keeps model inputs manageable.
- `FUTURE_LEN=96` evaluates the first 96 forecast steps, not the whole long horizon.
- `MAX_MODEL_CHAIN_LEN=2` means the exhaustive pass tests one-model and two-model residual chains.
- `ENABLE_LIGHT_XGBOOST=True` adds a small XGBoost model if the package is installed.
- Turn on heavier tree models later if the light pass looks useful.

In [ ]:
CONFIG_NAME = "electricity_H_long"  # Energy, hourly, long horizon

# Data slice. With start=0, stop=120, step=4 this gives about 30 samples.
N_START = 0
N_STOP = 120
STEP = 4

# Shape controls for CLP experiments.
HISTORY_LEN = 512
FUTURE_LEN = 96

# Exhaustive search controls.
RUN_EXHAUSTIVE = True
MAX_MODEL_CHAIN_LEN = 2  # 1 = single model; 2 = model -> residual model; 3 gets heavier.

# Cleaning policy.
MANDATORY_NORMALIZE = True
ENABLE_DETREND_AFTER_NORMALIZE = True

# Optional MCTS controls. Leave off while inspecting all small combinations.
RUN_MCTS = False
N_MCTS_ITERATIONS = 25
PUCT_C = 1.5

# Model vocabulary.
ENABLE_LIGHT_XGBOOST = True
ENABLE_TREE_MODELS = False  # heavier random forest token
ENABLE_XGBOOST = False      # heavier built-in xgboost token

SEED = 0

## 2. Load `electricity_H_long`

This uses `build_examples` and `uniform_length` from the existing kernel playground. Only histories are resized; futures are cropped to `FUTURE_LEN` in this notebook.

In [ ]:
raw = build_examples(
    config=CONFIG_NAME,
    start=N_START,
    stop=N_STOP,
    step=STEP,
    dataset_name=DATASETS["eval"],
)

fixed = uniform_length(
    raw,
    target_len=HISTORY_LEN,
    min_len=HISTORY_LEN // 2,
    keys=("history",),
    seed=SEED,
    verbose=True,
)

# Keep examples with enough future horizon.
fixed = [r for r in fixed if len(r["future"]) >= FUTURE_LEN]

H = np.stack([np.asarray(r["history"], dtype=np.float32) for r in fixed])
F = np.stack([np.asarray(r["future"][:FUTURE_LEN], dtype=np.float32) for r in fixed])

print(f"loaded raw examples: {len(raw)}")
print(f"usable fixed examples: {len(fixed)}")
print("H shape:", H.shape)
print("F shape:", F.shape)
assert H.ndim == 2 and F.ndim == 2
assert H.shape[0] == F.shape[0]
assert H.shape[0] >= 6, "MCTS/LOO needs a few samples; increase N_STOP or reduce STEP."

In [ ]:
def plot_example_grid(H, F, n=6):
    n = min(n, H.shape[0])
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.2 * rows), squeeze=False)
    for i in range(rows * cols):
        ax = axes[i // cols, i % cols]
        if i >= n:
            ax.set_visible(False)
            continue
        h = H[i]
        f = F[i]
        ax.plot(np.arange(-len(h), 0), h, color="0.55", lw=1.0, label="history")
        ax.plot(np.arange(len(f)), f, color="black", lw=1.2, label="future")
        ax.axvline(0, color="tab:blue", ls=":", lw=0.8)
        ax.set_title(f"sample {i}")
        ax.grid(True, alpha=0.2)
        if i == 0:
            ax.legend(fontsize=8)
    fig.tight_layout()
    return fig

plot_example_grid(H, F, n=6);

## 3. Build the CLP Grammar

This grammar is the computational language. Tokens are small operations, and edges say which operations may follow which.

The default grammar now enforces:

- `normalize` immediately after START
- optional `detrend_after_norm` after normalization
- feature extraction after either `normalize` or `detrend_after_norm`
- model tokens after features, with residual chaining when models repeat

The notebook-local `xgboost_light` token is deliberately small so it can participate in exhaustive search without making the first pass miserable.

In [ ]:
def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


class CleanDetrendAfterNormalize(Token):
    """Optional detrending step after mandatory normalization."""
    name = "detrend_after_norm"
    token_class = "cleaning"
    reads = ["cleaned"]
    writes = ["cleaned", "trend_slope_norm", "trend_intercept_norm"]
    description = "Remove a linear trend from already-normalized histories"

    def apply(self, state):
        X = state.features["cleaned"].astype(np.float32)
        n, d = X.shape
        t = np.arange(d, dtype=np.float32)
        t_mean = t.mean()
        denom = np.sum((t - t_mean) ** 2) + 1e-8

        detrended = np.zeros_like(X)
        slopes = np.zeros(n, dtype=np.float32)
        intercepts = np.zeros(n, dtype=np.float32)
        for i in range(n):
            x_mean = X[i].mean()
            slope = np.sum((t - t_mean) * (X[i] - x_mean)) / denom
            intercept = x_mean - slope * t_mean
            slopes[i] = slope
            intercepts[i] = intercept
            detrended[i] = X[i] - (slope * t + intercept)

        state.features["cleaned"] = detrended
        state.features["trend_slope_norm"] = slopes
        state.features["trend_intercept_norm"] = intercepts
        state.log_step(
            self.name,
            {"cleaned": X.shape},
            {
                "cleaned": detrended.shape,
                "trend_slope_norm": slopes.shape,
                "trend_intercept_norm": intercepts.shape,
            },
        )
        state.token_sequence.append(self.name)
        return state


class ModelXGBoostLight(Token):
    """Small XGBoost LOO residual model for quick exhaustive experiments."""
    name = "xgboost_light"
    token_class = "model"
    reads = ["model_input"]
    writes = []
    description = "Light XGBoost residual model - explicit LOO"

    def apply(self, state):
        import gc
        import xgboost as xgb

        X = state.features["model_input"].astype(np.float32)
        Y = state.current_target.astype(np.float32)
        n = X.shape[0]
        Y_loo = np.zeros_like(Y)

        for i in range(n):
            mask = np.ones(n, dtype=bool)
            mask[i] = False
            m = xgb.XGBRegressor(
                n_estimators=25,
                max_depth=2,
                learning_rate=0.08,
                subsample=0.85,
                colsample_bytree=0.8,
                reg_alpha=0.2,
                reg_lambda=2.0,
                min_child_weight=max(1, (n - 1) // 6),
                objective="reg:squarederror",
                tree_method="hist",
                verbosity=0,
                random_state=SEED,
            )
            m.fit(X[mask], Y[mask])
            Y_loo[i] = m.predict(X[i:i + 1])[0]
            del m

        state.push_prediction(Y_loo, self.name)
        state.log_step(
            self.name,
            {"model_input": X.shape, "current_target": Y.shape},
            {"prediction_stack[-1]": Y_loo.shape, "current_target": state.current_target.shape},
        )
        state.token_sequence.append(self.name)
        gc.collect()
        return state


def build_energy_grammar(enable_tree_models=False, enable_xgboost=False, enable_light_xgboost=True):
    grammar = Grammar()

    # Mandatory first step: normalize.
    grammar.register(CleanNormalize(), follows=["START"])

    feature_predecessors = ["normalize"]
    if ENABLE_DETREND_AFTER_NORMALIZE:
        grammar.register(CleanDetrendAfterNormalize(), follows=["normalize"])
        feature_predecessors.append("detrend_after_norm")

    # Feature tokens.
    feature_tokens = [FeatRaw(), FeatFFTEncode(), FeatLagFeatures()]
    for prev in feature_predecessors:
        for tok in feature_tokens:
            grammar.register(tok, follows=[prev])

    # Model tokens.
    model_tokens = [ModelKernelRBF()]
    if enable_light_xgboost:
        if module_available("xgboost"):
            model_tokens.append(ModelXGBoostLight())
        else:
            print("xgboost not installed; skipping xgboost_light token.")
    if enable_tree_models:
        model_tokens.append(ModelRandomForest())
    if enable_xgboost and module_available("xgboost"):
        model_tokens.append(ModelXGBoost())
    elif enable_xgboost:
        print("xgboost not installed; skipping built-in xgboost token.")

    feature_names = [t.name for t in feature_tokens]
    model_names = [t.name for t in model_tokens]
    model_follows = feature_names + model_names
    model_leads = model_names + ["STOP"]
    for tok in model_tokens:
        grammar.register(tok, follows=model_follows, leads_to=model_leads)

    grammar.register(StopToken(), follows=[])
    return grammar


grammar = build_energy_grammar(
    enable_tree_models=ENABLE_TREE_MODELS,
    enable_xgboost=ENABLE_XGBOOST,
    enable_light_xgboost=ENABLE_LIGHT_XGBOOST,
)
grammar

In [ ]:
print("tokens:", grammar.token_names)
print("edges:", grammar.graph.number_of_edges())
plot_grammar(grammar, title="CLP grammar for electricity_H_long")

## 4. Exhaustive Combinations and Residual Chains

Each chain now has the shape:

`normalize -> [detrend_after_norm] -> feature -> model [-> model ...] -> STOP`

Normalization is mandatory and first. Detrending is optional and happens after normalization. When models are sequenced, the existing graph state already does residual fitting:

- every model token reads `state.current_target`
- `state.current_target` starts as the normalized true future after `normalize`
- after a model predicts, `state.push_prediction(...)` updates it to `future - sum(predictions_so_far)`
- therefore the next model in the chain fits the residual left by previous models

This section enumerates every enabled combination up to `MAX_MODEL_CHAIN_LEN` and ranks them by MASE.

In [ ]:
from itertools import product

def run_chain(H, F, token_names, grammar, return_residual_norms=False):
    """Run a token chain. Sequential models automatically fit residuals via State.current_target."""
    state = State(H, F)
    residual_norms = [("initial_target", float(np.linalg.norm(state.current_target)))]

    for name in token_names:
        state = grammar.tokens[name].apply(state)
        if name == "normalize":
            residual_norms.append(("after_normalize", float(np.linalg.norm(state.current_target))))
        if grammar.tokens[name].token_class == "model":
            residual_norms.append((name, float(np.linalg.norm(state.current_target))))

    if not state.terminated:
        state = grammar.tokens["STOP"].apply(state)

    if return_residual_norms:
        return state, residual_norms
    return state

cleaning_prefixes = [["normalize"]]
if ENABLE_DETREND_AFTER_NORMALIZE and "detrend_after_norm" in grammar.tokens:
    cleaning_prefixes.append(["normalize", "detrend_after_norm"])

feature_names = ["feat_raw", "fft_encode", "feat_lag"]
model_names = ["kernel_rbf"]
if ENABLE_LIGHT_XGBOOST and "xgboost_light" in grammar.tokens:
    model_names.append("xgboost_light")
if ENABLE_TREE_MODELS and "random_forest" in grammar.tokens:
    model_names.append("random_forest")
if ENABLE_XGBOOST and "xgboost" in grammar.tokens:
    model_names.append("xgboost")

candidate_chains = []
for cleaning_prefix in cleaning_prefixes:
    for feature in feature_names:
        for depth in range(1, MAX_MODEL_CHAIN_LEN + 1):
            for model_seq in product(model_names, repeat=depth):
                candidate_chains.append([*cleaning_prefix, feature, *model_seq, "STOP"])

print(f"cleaning prefixes: {cleaning_prefixes}")
print(f"enabled models: {model_names}")
print(f"candidate chains: {len(candidate_chains)}")

exhaustive_results = []
if RUN_EXHAUSTIVE:
    for chain in candidate_chains:
        chain_str = " -> ".join(chain)
        try:
            s, residual_norms = run_chain(H, F, chain, grammar, return_residual_norms=True)
            exhaustive_results.append({
                "chain": chain_str,
                "tokens": chain,
                "mase": s.mase,
                "n_models": sum(grammar.tokens[t].token_class == "model" for t in chain if t in grammar.tokens),
                "residual_norms": residual_norms,
                "state": s,
                "error": None,
            })
        except Exception as exc:
            exhaustive_results.append({
                "chain": chain_str,
                "tokens": chain,
                "mase": np.inf,
                "n_models": sum(t in model_names for t in chain),
                "residual_norms": [],
                "state": None,
                "error": repr(exc),
            })

exhaustive_results = sorted(exhaustive_results, key=lambda r: r["mase"])
failed = [r for r in exhaustive_results if r["error"]]
print(f"finished exhaustive pass: {len(exhaustive_results) - len(failed)} ok, {len(failed)} failed")
print("
Top exhaustive chains")
print("-" * 130)
for row in exhaustive_results[:15]:
    print(f"MASE={row['mase']:8.4f} | models={row['n_models']} | {row['chain']}")

if failed:
    print("
First failures")
    for row in failed[:5]:
        print(row["chain"], "=>", row["error"])

best_exhaustive = exhaustive_results[0] if exhaustive_results else None
if best_exhaustive:
    print("
Best residual norm trace")
    for name, norm in best_exhaustive["residual_norms"]:
        print(f"{name:<20s} {norm:.4f}")

## 5. Optional MCTS

For now, `RUN_MCTS=False` by default because the exhaustive pass above shows every small combination directly. Turn it on when the grammar grows and exhaustive enumeration becomes too expensive.

In [ ]:
if RUN_MCTS:
    initial_state = State(H, F)
    mcts_results = mcts_search(
        grammar,
        initial_state,
        n_iterations=N_MCTS_ITERATIONS,
        puct_c=PUCT_C,
        verbose=True,
    )
    print("
Best MCTS chain:", mcts_results["best_chain"])
    print("Best MCTS MASE:", mcts_results["best_mase"])
else:
    mcts_results = None
    print("MCTS skipped. Set RUN_MCTS=True in the knobs cell to enable it.")

In [ ]:
if mcts_results is not None:
    print_mcts_tree(mcts_results["root"], max_depth=6)
else:
    print("No MCTS tree to print.")

## 6. Search History and Best Chains

This section reports either the MCTS trace, if MCTS ran, or the exhaustive ranking.

In [ ]:
if mcts_results is not None:
    history = mcts_results["history"]
    mase_curve = np.array([h["mase"] for h in history], dtype=float)
    best_so_far = np.minimum.accumulate(mase_curve)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(mase_curve, marker="o", lw=1, label="iteration MASE")
    ax.plot(best_so_far, marker="s", lw=2, label="best so far")
    ax.set_xlabel("MCTS iteration")
    ax.set_ylabel("MASE")
    ax.set_title("MCTS search progress")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
else:
    vals = np.array([r["mase"] for r in exhaustive_results if np.isfinite(r["mase"])], dtype=float)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(np.sort(vals), marker="o", lw=1)
    ax.set_xlabel("chain rank")
    ax.set_ylabel("MASE")
    ax.set_title("Exhaustive chain ranking")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()

In [ ]:
if mcts_results is not None:
    top_chains = sorted(mcts_results["all_chains"].items(), key=lambda kv: kv[1])[:10]
else:
    top_chains = [(r["chain"], r["mase"]) for r in exhaustive_results[:10]]

print("Top chains")
print("-" * 110)
for chain, mase in top_chains:
    print(f"MASE={mase:8.4f} | {chain}")

## 7. Re-run the Best Chain and Inspect Forecasts

This replays the best chain from MCTS if MCTS ran, otherwise from the exhaustive pass. The residual norm trace shows how much target remains after each model in a sequence.

In [ ]:
if mcts_results is not None:
    best_chain_str = mcts_results["best_chain"]
    best_source = "MCTS"
else:
    best_chain_str = exhaustive_results[0]["chain"]
    best_source = "exhaustive"

best_tokens = [t.strip() for t in best_chain_str.split("->")]
best_state, best_residual_norms = run_chain(H, F, best_tokens, grammar, return_residual_norms=True)

print("best source:", best_source)
print("best tokens:", best_tokens)
print("best MASE:", best_state.mase)
print("
Residual norm trace")
for name, norm in best_residual_norms:
    print(f"{name:<20s} {norm:.4f}")
print()
best_state.print_log()

In [ ]:
forecast = best_state.features["final_forecast"]

def plot_forecasts(H, F, forecast, n=6):
    n = min(n, H.shape[0])
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.4 * rows), squeeze=False)
    for i in range(rows * cols):
        ax = axes[i // cols, i % cols]
        if i >= n:
            ax.set_visible(False)
            continue
        h_tail = H[i, -min(160, H.shape[1]):]
        ax.plot(np.arange(-len(h_tail), 0), h_tail, color="0.65", lw=1.0, label="history")
        ax.plot(np.arange(F.shape[1]), F[i], color="black", lw=1.4, label="actual")
        ax.plot(np.arange(forecast.shape[1]), forecast[i], color="tab:orange", ls="--", lw=1.4, label="forecast")
        ax.axvline(0, color="tab:blue", ls=":", lw=0.8)
        ax.set_title(f"sample {i}")
        ax.grid(True, alpha=0.2)
        if i == 0:
            ax.legend(fontsize=8)
    fig.tight_layout()
    return fig

plot_forecasts(H, F, forecast, n=6);

## 8. GIEN-Style Graph Interpretation

Here GIEN means a graph interpretation / experiment-notebook view: inspect which tokens and branches look useful. With exhaustive search, this summarizes token frequency among the top chains. With MCTS, it also inspects tree visits.

In [ ]:
def flatten_tree(node, rows=None, prefix=()):
    if rows is None:
        rows = []
    for child in node.children.values():
        path = prefix + (child.name,)
        est_mase = (1.0 / child.avg_reward - 1.0) if child.avg_reward > 0 else float("inf")
        rows.append({
            "path": " -> ".join(path),
            "token": child.name,
            "visits": child.visits,
            "avg_reward": child.avg_reward,
            "est_mase": est_mase,
            "prior": child.prior,
        })
        flatten_tree(child, rows, path)
    return rows

if mcts_results is not None:
    tree_rows = flatten_tree(mcts_results["root"])
    tree_rows_sorted = sorted(tree_rows, key=lambda r: (-r["visits"], r["est_mase"]))
    print("Most visited MCTS tree nodes")
    print("-" * 120)
    for r in tree_rows_sorted[:20]:
        print(f"visits={r['visits']:3d} est_MASE={r['est_mase']:8.4f} prior={r['prior']:.3f} | {r['path']}")
else:
    tree_rows = []
    print("MCTS did not run; showing exhaustive token summaries in the next cell.")

In [ ]:
from collections import Counter

if mcts_results is not None:
    token_counts = Counter()
    for r in tree_rows:
        token_counts[r["token"]] += r["visits"]
    title = "Token visit mass across MCTS tree"
    xlabel = "visits"
else:
    top_k = min(15, len(exhaustive_results))
    token_counts = Counter()
    for r in exhaustive_results[:top_k]:
        token_counts.update([t for t in r["tokens"] if t != "STOP"])
    title = f"Token frequency in top {top_k} exhaustive chains"
    xlabel = "count in top chains"

items = token_counts.most_common()
labels = [k for k, _ in items]
values = [v for _, v in items]

fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(labels))))
ax.barh(labels[::-1], values[::-1], color="tab:blue", alpha=0.75)
ax.set_xlabel(xlabel)
ax.set_title(title)
ax.grid(True, axis="x", alpha=0.25)
fig.tight_layout()

## 9. Next Experiment Ideas

Good next moves after the first run:

- Increase `MAX_MODEL_CHAIN_LEN` to 3 after checking runtime.
- Increase `N_STOP` or reduce `STEP` to give the LOO models more examples.
- Try `HISTORY_LEN` values like 256, 512, 1024, and 2048.
- Compare `normalize -> feature -> ...` vs `normalize -> detrend_after_norm -> feature -> ...` in the exhaustive table.
- Keep `ENABLE_LIGHT_XGBOOST=True` for a small XGBoost model; disable it if runtime gets annoying.
- Turn on `ENABLE_TREE_MODELS=True` to enumerate RBF and random forest residual chains.
- Turn on `RUN_MCTS=True` when exhaustive search becomes too large.
- Add a new token for the masked RBF idea from `first_tests/utils/kernels.py`.
- Add decomposition tokens for CWT/KMD/EMD if the feature graph starts to plateau.